## Fazilka Flood Impact on cropland using Remote Sensing & GeoAI

### Year chosen : 2025

In [15]:
"""
Flood impact on agriculture - Gurdaspur district, Punjab (India), Aug-Sep 2025 floods.
Sentinel-1 SAR change detection + ESA WorldCover cropland.
Prints district stats (ha) and shows one map. Change the CONFIG block to reuse for another district/event.
"""
import ee
import geemap

ee.Initialize(project="ee-zuhailabdullah")  # <- change

# ------------------------------------------------------------------ CONFIG
COUNTRY, STATE, DISTRICT = "India", "Punjab", "Fazilka"   # FAO GAUL level-2 names
BEFORE_START, BEFORE_END = "2025-07-25", "2025-08-20"       # pre-flood reference
FLOOD_START,  FLOOD_END  = "2025-08-26", "2025-09-14"       # flood window (Ravi/Beas/Ujh surge)
ORBIT_PASS   = None       # 'ASCENDING' | 'DESCENDING' | None (None = more scenes in a short window)
DIFF_DB      = -3.0       # flood if VV drops by more than this (dB)...
WATER_DB     = -15.0      # ...and flood-window VV is below this (dB)
MIN_PATCH_PX = 8          # drop speckle-sized patches (10 m pixels)
PERM_WATER_OCC = 80       # JRC occurrence (%) treated as permanent water
MAX_SLOPE_DEG  = 5        # ignore steep terrain (radar shadow)
USE_STANDING_CROP_FILTER = True   # count only cropland that was green before the flood
NDVI_MIN = 0.3
# -----------------------------------------------------------------------


In [16]:
import geopandas as gpd

SHP_PATH = r"shapefiles\pb_districts.shp"   # <- your file
NAME_FIELD = "District"                          # <- the attribute column holding district names

gdf = gpd.read_file(SHP_PATH)
# print(gdf[NAME_FIELD].unique())                  # check the exact spelling, e.g. "Gurdaspur" vs "GURDASPUR"

gdf = gdf[gdf[NAME_FIELD].str.strip().str.lower() == DISTRICT.lower()]
gdf = gdf.to_crs(4326)                           # Earth Engine needs lat/lon
region = geemap.geopandas_to_ee(gdf[[NAME_FIELD, "geometry"]])
geom = region.geometry()
print("District features found:", region.size().getInfo(),
      "| boundary area (ha): %.0f" % (geom.area(1).divide(1e4).getInfo()))
# If this boundary looks wrong (e.g. includes/excludes Pathankot or Batala), upload your own
# district shapefile as an asset and set: region = ee.FeatureCollection("projects/.../assets/gurdaspur")

District features found: 1 | boundary area (ha): 287826


In [17]:


# ------------------------------------------------ Sentinel-1 flood mapping
def s1_composite(start, end, reducer):
    col = (ee.ImageCollection("COPERNICUS/S1_GRD")
           .filterBounds(geom)
           .filterDate(start, end)
           .filter(ee.Filter.eq("instrumentMode", "IW"))
           .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
           .select("VV"))
    if ORBIT_PASS:
        col = col.filter(ee.Filter.eq("orbitProperties_pass", ORBIT_PASS))
    col = col.map(lambda i: i.focal_median(50, "circle", "meters")
                  .copyProperties(i, ["system:time_start"]))
    return col.reduce(reducer).rename("VV"), col.size()

before, n_before = s1_composite(BEFORE_START, BEFORE_END, ee.Reducer.median())
after, n_after = s1_composite(FLOOD_START, FLOOD_END, ee.Reducer.percentile([10]))  # ~max extent
print("S1 scenes  before:", n_before.getInfo(), "| flood window:", n_after.getInfo())

diff = after.subtract(before)
flooded = diff.lt(DIFF_DB).And(after.lt(WATER_DB))

jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").unmask(0)
slope = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003"))
wc = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")

flooded = (flooded
           .updateMask(jrc.lt(PERM_WATER_OCC))     # remove permanent water (river channels, ponds)
           .updateMask(slope.lt(MAX_SLOPE_DEG))
           .updateMask(wc.neq(80))
           .selfMask())
flooded = flooded.updateMask(flooded.connectedPixelCount(MIN_PATCH_PX, True).gte(MIN_PATCH_PX))
flooded = flooded.unmask(0).rename("flooded").clip(geom)

S1 scenes  before: 14 | flood window: 14


In [18]:


# ------------------------------------------------ Cropland
cropland = wc.eq(40).rename("cropland")            # ESA WorldCover 2021 cropland
if USE_STANDING_CROP_FILTER:
    def s2_mask(img):
        scl = img.select("SCL")
        ok = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
        return img.updateMask(ok)
    ndvi = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(geom)
            .filterDate(BEFORE_START, BEFORE_END)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
            .map(s2_mask)
            .map(lambda i: i.normalizedDifference(["B8", "B4"]).rename("NDVI"))
            .max())
    cropland = cropland.And(ndvi.gt(NDVI_MIN).unmask(0)).rename("cropland")
cropland = cropland.clip(geom)
flooded_crop = flooded.And(cropland).rename("flooded_crop")

In [21]:
# ------------------------------------------------ District statistics (ha)
px_ha = ee.Image.pixelArea().divide(1e4)

def total_ha(img):
    d = img.multiply(px_ha).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=geom, scale=10,
        maxPixels=1e13, bestEffort=True, tileScale=8)
    return ee.Number(d.values().get(0))
wc_crop = wc.eq(40).clip(geom)
print("Boundary ha:            %.0f" % geom.area(1).divide(1e4).getInfo())
print("WorldCover cropland ha: %.0f" % total_ha(wc_crop).getInfo())
print("After NDVI filter ha:   %.0f" % total_ha(cropland).getInfo())
crop_ha, flood_ha, fcrop_ha = total_ha(cropland), total_ha(flooded), total_ha(flooded_crop)
stats = ee.Dictionary({
    "cropland_ha": crop_ha,
    "flooded_area_ha": flood_ha,
    "flooded_cropland_ha": fcrop_ha,
    "pct_cropland_flooded": fcrop_ha.divide(crop_ha).multiply(100),
    "pct_flood_on_cropland": fcrop_ha.divide(flood_ha).multiply(100),
}).getInfo()
print(f"\n{DISTRICT} | flood window {FLOOD_START} to {FLOOD_END}")
for k, v in stats.items():
    print(f"  {k:24s} {v:,.1f}")

Boundary ha:            287826
WorldCover cropland ha: 248978
After NDVI filter ha:   58161

Fazilka | flood window 2025-08-26 to 2025-09-14
  cropland_ha              58,160.8
  flooded_area_ha          10,095.7
  flooded_cropland_ha      2,212.9
  pct_cropland_flooded     3.8
  pct_flood_on_cropland    21.9


In [20]:
# ------------------------------------------------ Map (district only)
Map = geemap.Map()
Map.centerObject(region, 10)
Map.addLayer(region.style(**{"color": "black", "fillColor": "00000000", "width": 2}), {}, f"{DISTRICT} boundary")
Map.addLayer(cropland.selfMask(), {"palette": ["lightgreen"]}, "Cropland (pre-flood)")
Map.addLayer(flooded.selfMask(), {"palette": ["blue"]}, "Flooded area")
Map.addLayer(flooded_crop.selfMask(), {"palette": ["red"]}, "Flooded cropland")
Map

Map(center=[30.27669266547657, 74.16364452148217], controls=(WidgetControl(options=['position', 'transparent_b…

### Year chosen: 200X